In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Projects_AI_DS/bearing_fault/IMS.zip"
extract_path = "/content/mydata"

# Create folder if not exists
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))


Extracted files: ['IMS', '__MACOSX']


In [3]:
!apt-get install unrar
!unrar x /content/mydata/IMS/1st_test.rar /content/mydata/IMS/
# !unrar x /content/mydata/IMS/2nd_test.rar /content/mydata/IMS/
# !unrar x /content/mydata/IMS/3rd_test.rar /content/mydata/IMS/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/mydata/IMS/1st_test.rar

Creating    /content/mydata/IMS/1st_test                              OK
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.06.24               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.09.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.14.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.19.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.24.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.29.13               0%  OK 
Extracting  /content/mydata/IMS

file_level

In [ ]:
# ================= FAST IMS BEARING FAULT DETECTION =================
# ================= WINDOW = 1024 | STRIDE = 512 =================
# ================= LEAKAGE-FREE EVALUATION =================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from lightgbm import LGBMClassifier

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1200

# -------------------------------------------------------
# FAST WINDOWING
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FAST FEATURE EXTRACTION
# -------------------------------------------------------

def extract_features(w):

    # w shape -> (N, 1024, 2)

    w = w.astype(np.float32)

    # ---------- RMS ----------
    rms = np.sqrt(np.mean(w * w, axis=1))

    # ---------- STD ----------
    std = np.std(w, axis=1)

    # ---------- PEAK ----------
    peak = np.max(np.abs(w), axis=1)

    # ---------- PEAK TO RMS ----------
    pr = peak / (rms + 1e-8)

    # ---------- TREND ----------
    t = np.arange(rms.shape[0], dtype=np.float32)

    t_mean = t.mean()
    t_var = np.sum((t - t_mean) ** 2) + 1e-8

    slopes = []

    for c in range(2):

        y = rms[:, c]

        y_mean = y.mean()

        slope = np.sum((t - t_mean) * (y - y_mean)) / t_var

        slopes.append(slope)

    # ---------- DAMAGE ----------
    damage = np.sum(np.abs(np.diff(rms.mean(axis=1))))

    # ---------- FINAL FEATURES ----------
    feats = np.concatenate([

        rms.mean(axis=0),
        rms.std(axis=0),

        std.mean(axis=0),
        std.std(axis=0),

        peak.mean(axis=0),
        peak.std(axis=0),

        pr.mean(axis=0),
        pr.std(axis=0),

        np.array(slopes),

        np.array([damage])

    ])

    return feats

# -------------------------------------------------------
# BUILD DATASET
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X = []
y = []
groups = []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4,5,6,7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    # ---------------- WINDOWING ----------------

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    # ---------------- FEATURES ----------------

    X.append(extract_features(w3))
    y.append(0 if i < HEALTHY_END else 1)
    groups.append(i)

    X.append(extract_features(w4))
    y.append(0 if i < HEALTHY_END else 2)
    groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:")
print(np.unique(y, return_counts=True))

# -------------------------------------------------------
# LEAKAGE-FREE EVALUATION
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

all_preds = []
all_true = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups)
):

    print(f"\n========== Fold {fold+1} ==========")

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # ---------------- SCALING ----------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ---------------- MODEL ----------------

    model = LGBMClassifier(
        n_estimators=100,      # reduced
        learning_rate=0.08,    # faster convergence
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,             # full CPU usage
        verbosity=-1
    )

    # ---------------- TRAIN ----------------

    model.fit(X_train, y_train)

    # ---------------- PREDICT ----------------

    preds = model.predict(X_test)

    all_preds.extend(preds)
    all_true.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# FINAL RESULT
# -------------------------------------------------------

print("\n================ FINAL RESULT ================")
print("Leakage-Free Evaluation using GroupKFold\n")

print(classification_report(all_true, all_preds))

Processed 0/2156 files
Processed 200/2156 files
Processed 400/2156 files
Processed 600/2156 files
Processed 800/2156 files
Processed 1000/2156 files
Processed 1200/2156 files
Processed 1400/2156 files
Processed 1600/2156 files
Processed 1800/2156 files
Processed 2000/2156 files

Feature Shape: (4312, 19)
Class Distribution:
(array([0, 1, 2]), array([2400,  956,  956]))

========== Fold 1 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 2 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 3 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 4 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 5 ==========
Fold Completed

================ FINAL RESULT ================
Leakage-Free Evaluation using GroupKFold

              precision    recall  f1-score   support

           0       0.90      0.90      0.90      2400
           1       0.76      0.80      0.78       956
           2       1.00      0.96      0.97       956

    accuracy                           0.89      4312
   macro avg       0.88      0.88      0.88      4312
weighted avg       0.89      0.89      0.89      4312



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


window_level

In [ ]:
# ================= FAST WINDOW-LEVEL IMS BEARING FAULT DETECTION =================
# ================= WINDOW = 1024 | STRIDE = 512 =================
# ================= LEAKAGE-FREE EVALUATION =================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from lightgbm import LGBMClassifier

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1600

# -------------------------------------------------------
# WINDOWING (FAST)
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FEATURE EXTRACTION
# -------------------------------------------------------

def extract_features(window):

    window = window.astype(np.float32)

    # ---------- basic stats ----------
    rms = np.sqrt(np.mean(window ** 2, axis=0))
    std = np.std(window, axis=0)
    peak = np.max(np.abs(window), axis=0)
    pr = peak / (rms + 1e-8)

    # ---------- trend ----------
    t = np.arange(window.shape[0], dtype=np.float32)

    slopes = []
    for c in range(window.shape[1]):
        y = window[:, c]
        slope = np.sum((t - t.mean()) * (y - y.mean())) / (
            np.sum((t - t.mean())**2) + 1e-8
        )
        slopes.append(slope)

    # ---------- damage ----------
    damage = np.sum(np.abs(np.diff(window.mean(axis=1))))

    # ONLY KEEP:
    # rms, std, peak, pr + slope + damage

    return np.concatenate([
        rms,
        std,
        peak,
        pr,
        np.array(slopes),
        np.array([damage])
    ])

# -------------------------------------------------------
# BUILD DATASET (WINDOW LEVEL)
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X = []
y = []
groups = []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4, 5, 6, 7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    label_b3 = 0 if i < HEALTHY_END else 1
    label_b4 = 0 if i < HEALTHY_END else 2

    # -------- window-level append --------

    for w in w3:
        X.append(extract_features(w))
        y.append(label_b3)
        groups.append(i)

    for w in w4:
        X.append(extract_features(w))
        y.append(label_b4)
        groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:", np.unique(y, return_counts=True))

# -------------------------------------------------------
# LEAKAGE-FREE EVALUATION
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

all_preds = []
all_true = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.08,
        num_leaves=31,
        class_weight={0:1, 1:3, 2:1},
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    all_preds.extend(preds)
    all_true.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# FINAL RESULT
# -------------------------------------------------------

print("\n================ FINAL RESULT ================")
print(classification_report(all_true, all_preds, digits=4))

Processed 0/2156 files
Processed 200/2156 files
Processed 400/2156 files
Processed 600/2156 files
Processed 800/2156 files
Processed 1000/2156 files
Processed 1200/2156 files
Processed 1400/2156 files
Processed 1600/2156 files
Processed 1800/2156 files
Processed 2000/2156 files

Feature Shape: (168168, 11)
Class Distribution: (array([0, 1, 2]), array([124800,  21684,  21684]))

========== Fold 1 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 2 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 3 ==========


...

Fold Completed

========== Fold 4 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 5 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

================ FINAL RESULT ================
              precision    recall  f1-score   support

           0     0.9487    0.9248    0.9366    124800
           1     0.6379    0.7434    0.6866     21684
           2     0.9794    0.9593    0.9692     21684

    accuracy                         0.9059    168168
   macro avg     0.8553    0.8758    0.8642    168168
weighted avg     0.9126    0.9059    0.9086    168168



In [ ]:
# ================= FAST WINDOW-LEVEL IMS BEARING FAULT DETECTION =================
# ================= WINDOW = 1024 | STRIDE = 512 =================
# ================= LEAKAGE-FREE EVALUATION =================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from lightgbm import LGBMClassifier

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1600

# -------------------------------------------------------
# WINDOWING (FAST)
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FEATURE EXTRACTION (IMPROVED FOR CLASS 1)
# -------------------------------------------------------

def extract_features(window):

    window = window.astype(np.float32)

    # ---------- basic stats ----------
    rms = np.sqrt(np.mean(window ** 2, axis=0))
    std = np.std(window, axis=0)
    peak = np.max(np.abs(window), axis=0)
    pr = peak / (rms + 1e-8)

    # ---------- trend ----------
    t = np.arange(window.shape[0], dtype=np.float32)

    slopes = []
    for c in range(window.shape[1]):
        y = window[:, c]
        slope = np.sum((t - t.mean()) * (y - y.mean())) / (np.sum((t - t.mean())**2) + 1e-8)
        slopes.append(slope)

    # ---------- IMPORTANT (class 1 improvement features) ----------

    # 1. signal roughness (early fault sensitivity)
    roughness = np.mean(np.diff(window, axis=0) ** 2)

    # 2. high frequency energy (inner race indicator)
    fft = np.abs(np.fft.rfft(window, axis=0))
    hf_energy = np.sum(fft[20:], axis=0).mean() / (np.sum(fft) + 1e-8)

    # ---------- damage ----------
    damage = np.sum(np.abs(np.diff(window.mean(axis=1))))

    return np.concatenate([
        rms,
        std,
        peak,
        pr,
        np.array(slopes),
        np.array([damage, roughness, hf_energy])
    ])

# -------------------------------------------------------
# BUILD DATASET (WINDOW LEVEL)
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X = []
y = []
groups = []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4, 5, 6, 7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    label_b3 = 0 if i < HEALTHY_END else 1
    label_b4 = 0 if i < HEALTHY_END else 2

    # -------- window-level append --------
    for w in w3:
        X.append(extract_features(w))
        y.append(label_b3)
        groups.append(i)

    for w in w4:
        X.append(extract_features(w))
        y.append(label_b4)
        groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:", np.unique(y, return_counts=True))

# -------------------------------------------------------
# LEAKAGE-FREE EVALUATION
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

all_preds = []
all_true = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.08,
        num_leaves=31,
        class_weight={0:1, 1:3, 2:1},   # 🔥 improved class-1 focus
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    all_preds.extend(preds)
    all_true.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# FINAL RESULT
# -------------------------------------------------------

print("\n================ FINAL RESULT ================")
print(classification_report(all_true, all_preds, digits=4))

Processed 0/2156 files
Processed 200/2156 files
Processed 400/2156 files
Processed 600/2156 files
Processed 800/2156 files
Processed 1000/2156 files
Processed 1200/2156 files
Processed 1400/2156 files
Processed 1600/2156 files
Processed 1800/2156 files
Processed 2000/2156 files

Feature Shape: (168168, 13)
Class Distribution: (array([0, 1, 2]), array([124800,  21684,  21684]))

========== Fold 1 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 2 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 3 ==========


...

Fold Completed

========== Fold 4 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 5 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

================ FINAL RESULT ================
              precision    recall  f1-score   support

           0     0.9576    0.9421    0.9498    124800
           1     0.7123    0.7919    0.7500     21684
           2     0.9801    0.9615    0.9707     21684

    accuracy                         0.9253    168168
   macro avg     0.8833    0.8985    0.8902    168168
weighted avg     0.9288    0.9253    0.9267    168168



In [ ]:
# ================= FAST WINDOW-LEVEL IMS BEARING FAULT DETECTION =================
# ================= WINDOW = 1024 | STRIDE = 512 =================
# ================= LEAKAGE-FREE EVALUATION =================

import os
import numpy as np
import pandas as pd

from scipy.signal import hilbert

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from lightgbm import LGBMClassifier

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1600

# -------------------------------------------------------
# WINDOWING (FAST)
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FEATURE EXTRACTION (IMPROVED FOR CLASS 1)
# -------------------------------------------------------

def extract_features(window):

    window = window.astype(np.float32)

    # ---------- basic stats ----------
    rms = np.sqrt(np.mean(window ** 2, axis=0))
    std = np.std(window, axis=0)
    peak = np.max(np.abs(window), axis=0)
    pr = peak / (rms + 1e-8)

    # ---------- trend ----------
    t = np.arange(window.shape[0], dtype=np.float32)

    slopes = []

    for c in range(window.shape[1]):

        y = window[:, c]

        slope = np.sum((t - t.mean()) * (y - y.mean())) / (
            np.sum((t - t.mean())**2) + 1e-8
        )

        slopes.append(slope)

    # ---------- IMPORTANT FEATURES ----------

    # 1. signal roughness
    roughness = np.mean(np.diff(window, axis=0) ** 2)

    # 2. high frequency energy
    fft = np.abs(np.fft.rfft(window, axis=0))

    hf_energy = (
        np.sum(fft[20:], axis=0).mean()
        / (np.sum(fft) + 1e-8)
    )

    # 3. envelope energy
    env = np.abs(hilbert(window, axis=0))
    env_energy = np.mean(env ** 2)

    # 4. crest factor
    crest = np.mean(peak / (std + 1e-8))

    # ---------- damage ----------
    damage = np.sum(np.abs(np.diff(window.mean(axis=1))))

    # ---------- FINAL FEATURES ----------

    return np.concatenate([

        rms,
        std,
        peak,
        pr,

        np.array(slopes),

        np.array([
            damage,
            roughness,
            hf_energy,
            env_energy,
            crest
        ])

    ])

# -------------------------------------------------------
# BUILD DATASET (WINDOW LEVEL)
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X = []
y = []
groups = []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4, 5, 6, 7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    label_b3 = 0 if i < HEALTHY_END else 1
    label_b4 = 0 if i < HEALTHY_END else 2

    # -------- window-level append --------

    for w in w3:
        X.append(extract_features(w))
        y.append(label_b3)
        groups.append(i)

    for w in w4:
        X.append(extract_features(w))
        y.append(label_b4)
        groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:", np.unique(y, return_counts=True))

# -------------------------------------------------------
# LEAKAGE-FREE EVALUATION
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

all_preds = []
all_true = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.08,
        num_leaves=31,
        class_weight={0:1, 1:3, 2:1},
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    all_preds.extend(preds)
    all_true.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# FINAL RESULT
# -------------------------------------------------------

print("\n================ FINAL RESULT ================")
print(classification_report(all_true, all_preds, digits=4))

Processed 0/2156 files
Processed 200/2156 files
Processed 400/2156 files
Processed 600/2156 files
Processed 800/2156 files
Processed 1000/2156 files
Processed 1200/2156 files
Processed 1400/2156 files
Processed 1600/2156 files
Processed 1800/2156 files
Processed 2000/2156 files

Feature Shape: (168168, 15)
Class Distribution: (array([0, 1, 2]), array([124800,  21684,  21684]))

========== Fold 1 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 2 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 3 ==========


...

Fold Completed

========== Fold 4 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

========== Fold 5 ==========


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold Completed

================ FINAL RESULT ================
              precision    recall  f1-score   support

           0     0.9575    0.9416    0.9495    124800
           1     0.7107    0.7920    0.7491     21684
           2     0.9798    0.9616    0.9706     21684

    accuracy                         0.9249    168168
   macro avg     0.8827    0.8984    0.8898    168168
weighted avg     0.9286    0.9249    0.9264    168168



In [ ]:
# ================= WINDOW-LEVEL IMS PIPELINE =================
# ===== LGBM FEATURE SELECTION -> XGBOOST + RF ==============

import os
import numpy as np
import pandas as pd

from scipy.signal import hilbert

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1600

TOP_K = 10   # top important features

# -------------------------------------------------------
# WINDOWING
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FEATURE EXTRACTION
# -------------------------------------------------------

def extract_features(window):

    window = window.astype(np.float32)

    # ---------- basic stats ----------
    rms = np.sqrt(np.mean(window ** 2, axis=0))
    std = np.std(window, axis=0)
    peak = np.max(np.abs(window), axis=0)
    pr = peak / (rms + 1e-8)

    # ---------- trend ----------
    t = np.arange(window.shape[0], dtype=np.float32)

    slopes = []

    for c in range(window.shape[1]):

        y = window[:, c]

        slope = np.sum((t - t.mean()) * (y - y.mean())) / (
            np.sum((t - t.mean())**2) + 1e-8
        )

        slopes.append(slope)

    # ---------- extra features ----------

    roughness = np.mean(np.diff(window, axis=0) ** 2)

    fft = np.abs(np.fft.rfft(window, axis=0))

    hf_energy = (
        np.sum(fft[20:], axis=0).mean()
        / (np.sum(fft) + 1e-8)
    )

    env = np.abs(hilbert(window, axis=0))
    env_energy = np.mean(env ** 2)

    crest = np.mean(peak / (std + 1e-8))

    damage = np.sum(np.abs(np.diff(window.mean(axis=1))))

    return np.concatenate([

        rms,
        std,
        peak,
        pr,

        np.array(slopes),

        np.array([
            damage,
            roughness,
            hf_energy,
            env_energy,
            crest
        ])

    ])

# -------------------------------------------------------
# BUILD DATASET
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X = []
y = []
groups = []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4, 5, 6, 7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    label_b3 = 0 if i < HEALTHY_END else 1
    label_b4 = 0 if i < HEALTHY_END else 2

    for w in w3:
        X.append(extract_features(w))
        y.append(label_b3)
        groups.append(i)

    for w in w4:
        X.append(extract_features(w))
        y.append(label_b4)
        groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:", np.unique(y, return_counts=True))

# -------------------------------------------------------
# GROUP KFOLD
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

# =======================================================
# XGBOOST RESULTS
# =======================================================

xgb_preds = []
xgb_true = []

# =======================================================
# RF RESULTS
# =======================================================

rf_preds = []
rf_true = []

# -------------------------------------------------------
# CV LOOP
# -------------------------------------------------------

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # ---------------- SCALE ----------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ---------------------------------------------------
    # STAGE 1 -> LIGHTGBM FEATURE IMPORTANCE
    # ---------------------------------------------------

    selector_model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.08,
        num_leaves=31,
        class_weight={0:1, 1:3, 2:1},
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    selector_model.fit(X_train, y_train)

    importances = selector_model.feature_importances_

    top_idx = np.argsort(importances)[-TOP_K:]

    X_train_sel = X_train[:, top_idx]
    X_test_sel = X_test[:, top_idx]

    print("Selected Features:", top_idx)

    # ---------------------------------------------------
    # STAGE 2 -> XGBOOST
    # ---------------------------------------------------

    xgb_model = XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=3,
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    )

    xgb_model.fit(X_train_sel, y_train)

    xgb_pred = xgb_model.predict(X_test_sel)

    xgb_preds.extend(xgb_pred)
    xgb_true.extend(y_test)

    # ---------------------------------------------------
    # STAGE 3 -> RANDOM FOREST
    # ---------------------------------------------------

    rf_model = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

    rf_model.fit(X_train_sel, y_train)

    rf_pred = rf_model.predict(X_test_sel)

    rf_preds.extend(rf_pred)
    rf_true.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# FINAL RESULTS
# -------------------------------------------------------

print("\n================ XGBOOST RESULT ================")
print(classification_report(xgb_true, xgb_preds, digits=4))

print("\n================ RANDOM FOREST RESULT ================")
print(classification_report(rf_true, rf_preds, digits=4))

Processed 0/2156 files
Processed 200/2156 files
Processed 400/2156 files
Processed 600/2156 files
Processed 800/2156 files
Processed 1000/2156 files
Processed 1200/2156 files
Processed 1400/2156 files
Processed 1600/2156 files
Processed 1800/2156 files
Processed 2000/2156 files

Feature Shape: (168168, 15)
Class Distribution: (array([0, 1, 2]), array([124800,  21684,  21684]))

========== Fold 1 ==========
Selected Features: [ 6  4  1 14 12  2 13  3 10 11]
Fold Completed

========== Fold 2 ==========
Selected Features: [ 4  6  1 14 12  2 13  3 10 11]
Fold Completed

========== Fold 3 ==========
Selected Features: [ 5  1  6 14 12  2 13  3 10 11]
Fold Completed

========== Fold 4 ==========
Selected Features: [ 4  1  6 14 12  2 13  3 10 11]
Fold Completed

========== Fold 5 ==========
Selected Features: [ 6  4  1 14 12 13  2  3 10 11]
Fold Completed

================ XGBOOST RESULT ================
              precision    recall  f1-score   support

           0     0.9397    0.9819  

In [ ]:
# ================= FAST WINDOW-LEVEL IMS BEARING FAULT DETECTION =================
# ================= LIGHTGBM FEATURE SELECTION + SVM + NAIVE BAYES =================
# ================= WINDOW = 1024 | STRIDE = 512 =================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1600

# -------------------------------------------------------
# WINDOWING
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FEATURE EXTRACTION
# -------------------------------------------------------

def extract_features(window):

    window = window.astype(np.float32)

    rms = np.sqrt(np.mean(window ** 2, axis=0))
    std = np.std(window, axis=0)
    peak = np.max(np.abs(window), axis=0)
    pr = peak / (rms + 1e-8)

    t = np.arange(window.shape[0], dtype=np.float32)

    slopes = []
    for c in range(window.shape[1]):
        y = window[:, c]
        slope = np.sum((t - t.mean()) * (y - y.mean())) / (
            np.sum((t - t.mean())**2) + 1e-8
        )
        slopes.append(slope)

    roughness = np.mean(np.diff(window, axis=0) ** 2)

    fft = np.abs(np.fft.rfft(window, axis=0))
    hf_energy = np.sum(fft[20:], axis=0).mean() / (np.sum(fft) + 1e-8)

    damage = np.sum(np.abs(np.diff(window.mean(axis=1))))

    return np.concatenate([
        rms,
        std,
        peak,
        pr,
        np.array(slopes),
        np.array([damage, roughness, hf_energy])
    ])

# -------------------------------------------------------
# BUILD DATASET
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X, y, groups = [], [], []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4, 5, 6, 7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    label_b3 = 0 if i < HEALTHY_END else 1
    label_b4 = 0 if i < HEALTHY_END else 2

    for w in w3:
        X.append(extract_features(w))
        y.append(label_b3)
        groups.append(i)

    for w in w4:
        X.append(extract_features(w))
        y.append(label_b4)
        groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:", np.unique(y, return_counts=True))

# -------------------------------------------------------
# GROUP K-FOLD
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

all_preds_svm, all_true_svm = [], []
all_preds_nb, all_true_nb = [], []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # SCALE (important for SVM)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ---------------- SVM ----------------
    svm = SVC(
        kernel="rbf",
        C=5,
        gamma="scale",
        class_weight="balanced"
    )

    svm.fit(X_train, y_train)
    pred_svm = svm.predict(X_test)

    all_preds_svm.extend(pred_svm)
    all_true_svm.extend(y_test)

    # ---------------- NAIVE BAYES ----------------
    nb = GaussianNB()
    nb.fit(X_train, y_train)

    pred_nb = nb.predict(X_test)

    all_preds_nb.extend(pred_nb)
    all_true_nb.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# RESULTS
# -------------------------------------------------------

print("\n================ SVM RESULT ================")
print(classification_report(all_true_svm, all_preds_svm, digits=4))

print("\n================ NAIVE BAYES RESULT ================")
print(classification_report(all_true_nb, all_preds_nb, digits=4))

Processed 0/2156 files
Processed 200/2156 files
Processed 400/2156 files
Processed 600/2156 files
Processed 800/2156 files
Processed 1000/2156 files
Processed 1200/2156 files
Processed 1400/2156 files
Processed 1600/2156 files
Processed 1800/2156 files
Processed 2000/2156 files

Feature Shape: (168168, 13)
Class Distribution: (array([0, 1, 2]), array([124800,  21684,  21684]))

========== Fold 1 ==========


In [ ]:
# ================= WINDOW-LEVEL IMS PIPELINE =================
# ===== LGBM FEATURE SELECTION -> SVM + NAIVE BAYES ==========

import os
import numpy as np
import pandas as pd

from scipy.signal import hilbert

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------

data_dir = "/content/mydata/IMS/1st_test"

window_size = 1024
stride = 512

HEALTHY_END = 1600

TOP_K = 10

# -------------------------------------------------------
# WINDOWING
# -------------------------------------------------------

def get_windows(signal):

    return np.lib.stride_tricks.sliding_window_view(
        signal,
        window_shape=(window_size, signal.shape[1])
    )[::stride, 0]

# -------------------------------------------------------
# FEATURE EXTRACTION
# -------------------------------------------------------

def extract_features(window):

    window = window.astype(np.float32)

    # ---------- basic stats ----------
    rms = np.sqrt(np.mean(window ** 2, axis=0))
    std = np.std(window, axis=0)
    peak = np.max(np.abs(window), axis=0)
    pr = peak / (rms + 1e-8)

    # ---------- trend ----------
    t = np.arange(window.shape[0], dtype=np.float32)

    slopes = []

    for c in range(window.shape[1]):

        y = window[:, c]

        slope = np.sum((t - t.mean()) * (y - y.mean())) / (
            np.sum((t - t.mean())**2) + 1e-8
        )

        slopes.append(slope)

    # ---------- extra features ----------

    roughness = np.mean(np.diff(window, axis=0) ** 2)

    fft = np.abs(np.fft.rfft(window, axis=0))

    hf_energy = (
        np.sum(fft[20:], axis=0).mean()
        / (np.sum(fft) + 1e-8)
    )

    env = np.abs(hilbert(window, axis=0))
    env_energy = np.mean(env ** 2)

    crest = np.mean(peak / (std + 1e-8))

    damage = np.sum(np.abs(np.diff(window.mean(axis=1))))

    return np.concatenate([

        rms,
        std,
        peak,
        pr,

        np.array(slopes),

        np.array([
            damage,
            roughness,
            hf_energy,
            env_energy,
            crest
        ])

    ])

# -------------------------------------------------------
# BUILD DATASET
# -------------------------------------------------------

files = sorted(os.listdir(data_dir))

X = []
y = []
groups = []

for i, file_name in enumerate(files):

    path = os.path.join(data_dir, file_name)

    signal = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[4, 5, 6, 7]
    ).values.astype(np.float32)

    B3 = signal[:, 0:2]
    B4 = signal[:, 2:4]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    label_b3 = 0 if i < HEALTHY_END else 1
    label_b4 = 0 if i < HEALTHY_END else 2

    for w in w3:
        X.append(extract_features(w))
        y.append(label_b3)
        groups.append(i)

    for w in w4:
        X.append(extract_features(w))
        y.append(label_b4)
        groups.append(i)

    if i % 200 == 0:
        print(f"Processed {i}/{len(files)} files")

# -------------------------------------------------------
# FINAL DATA
# -------------------------------------------------------

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("\nFeature Shape:", X.shape)
print("Class Distribution:", np.unique(y, return_counts=True))

# -------------------------------------------------------
# GROUP KFOLD
# -------------------------------------------------------

gkf = GroupKFold(n_splits=5)

# =======================================================
# SVM RESULTS
# =======================================================

svm_preds = []
svm_true = []

# =======================================================
# NAIVE BAYES RESULTS
# =======================================================

nb_preds = []
nb_true = []

# -------------------------------------------------------
# CV LOOP
# -------------------------------------------------------

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # ---------------- SCALE ----------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ---------------------------------------------------
    # STAGE 1 -> LIGHTGBM FEATURE IMPORTANCE
    # ---------------------------------------------------

    selector_model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.08,
        num_leaves=31,
        class_weight={0:1, 1:3, 2:1},
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    selector_model.fit(X_train, y_train)

    importances = selector_model.feature_importances_

    top_idx = np.argsort(importances)[-TOP_K:]

    X_train_sel = X_train[:, top_idx]
    X_test_sel = X_test[:, top_idx]

    print("Selected Features:", top_idx)

    # ---------------------------------------------------
    # STAGE 2 -> SVM
    # ---------------------------------------------------

    svm_model = SVC(
        kernel="rbf",
        C=5,
        gamma="scale",
        class_weight="balanced"
    )

    svm_model.fit(X_train_sel, y_train)

    svm_pred = svm_model.predict(X_test_sel)

    svm_preds.extend(svm_pred)
    svm_true.extend(y_test)

    # ---------------------------------------------------
    # STAGE 3 -> NAIVE BAYES
    # ---------------------------------------------------

    nb_model = GaussianNB()

    nb_model.fit(X_train_sel, y_train)

    nb_pred = nb_model.predict(X_test_sel)

    nb_preds.extend(nb_pred)
    nb_true.extend(y_test)

    print("Fold Completed")

# -------------------------------------------------------
# FINAL RESULTS
# -------------------------------------------------------

print("\n================ SVM RESULT ================")
print(classification_report(svm_true, svm_preds, digits=4))

print("\n================ NAIVE BAYES RESULT ================")
print(classification_report(nb_true, nb_preds, digits=4))